# Long-video memory: interactive demo

Download or select a video → transcribe and perceive it → inspect C3 memory updates →
ask a question at a selected time → inspect retrieved records and play supporting clips.

This notebook runs from the GitHub repository on Windows, Linux and Kaggle. No paths
from the author's computer are required. Enable Internet for initial setup, downloads
and model weights. Start with `local_small` and 180 seconds; use `kaggle` and
`DURATION_S = None` for a full-video run on two T4 GPUs.


## 1. Clone the repository and install dependencies

The cell reuses this checkout if available; otherwise it clones the demo branch from GitHub.
It never resets or switches an existing checkout. If installation changes packages already
loaded by your kernel, restart the kernel and run the notebook again.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/good-margaret/long_video_memory.git"
BRANCH = "codex/interactive-notebook-demo"
roots = [Path.cwd(), Path.cwd() / "long_video_memory", *Path.cwd().parents]
if Path("/kaggle/working").is_dir():
    roots.insert(0, Path("/kaggle/working/long_video_memory"))
REPO_ROOT = next((p for p in roots if (p / "src/memvid/demo/bootstrap.py").is_file()), None)
if REPO_ROOT is None:
    base = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
    REPO_ROOT = base / "long_video_memory"
    if REPO_ROOT.exists():
        raise RuntimeError("The target folder exists but has no demo bootstrap. Use the demo branch or an empty target.")
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_ROOT)], check=True)
REPO_ROOT = REPO_ROOT.resolve()
os.chdir(REPO_ROOT)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".[models,memory,asr,notebook]",
                       "torch>=2.4", "transformers==5.2.0", "accelerate>=1,<2"])
print("Project:", REPO_ROOT)


## 2. Media tools and optional YouTube download

FFmpeg and FFprobe are discovered or downloaded into `.cache/media-tools/`.
On Windows the managed binaries avoid incompatible Anaconda executables. On Linux/macOS
working system binaries are reused. Setup changes this kernel's environment, not the system PATH.

Set `DOWNLOAD_YOUTUBE = True` to download **video with audio** into `data/youtube/raw/`.
The default resolution cap is 720p to avoid a large 4K download; change it to 1080 or
`None` if needed. This cap is recorded with the source metadata and changes the visual input.
For comparisons use the same source and observations as the experiment you compare against.

The downloader uses `yt-dlp[default]` and discovers a supported Node/Deno runtime,
downloading portable Deno if necessary. YouTube can require sign-in or block a cloud IP;
in that case download on your computer and upload the media as Kaggle input. No browser
cookies or account credentials are read automatically.


In [ ]:
from memvid.demo.bootstrap import setup_media_tools, download_youtube

MEDIA_TOOLS = setup_media_tools(REPO_ROOT / ".cache" / "media-tools")
YOUTUBE_URL = "https://www.youtube.com/watch?v=KbahC-QCKU8"
DOWNLOAD_YOUTUBE = True
MAX_VIDEO_HEIGHT = 720  # 1080 or None for a higher-resolution source
RAW_DIR = REPO_ROOT / "data" / "youtube" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

if DOWNLOAD_YOUTUBE:
    VIDEO_PATH = str(download_youtube(YOUTUBE_URL, RAW_DIR, tools=MEDIA_TOOLS,
                     max_height=MAX_VIDEO_HEIGHT, cache_dir=REPO_ROOT / ".cache" / "js-runtime"))
    AUDIO_PATH = ""  # Downloaded video already contains audio.
else:
    VIDEO_PATH = str(RAW_DIR / "KbahC-QCKU8.h720.mp4")  # Or a /kaggle/input/... path.
    AUDIO_PATH = ""  # Set a separate .m4a/.wav path if your video is silent.
print("Video:", VIDEO_PATH)
print("Separate audio:", AUDIO_PATH or "using the video's audio stream")


## 3. Processing settings

`local_small`: CPU, SmolVLM-500M visual descriptions, Qwen3-0.6B for structured observations,
memory operations and QA, Whisper tiny.en with int8. Models run in separate worker processes
to release memory between stages. The small models can repeat, invent or omit details;
their outputs remain visible for inspection.

`kaggle`: Qwen3-VL-4B-Instruct, float16 with balanced placement across the two T4 GPUs,
and Whisper large-v3-turbo in a separate stage. This is a different model condition,
not an accuracy-equivalent local run. The full Kaggle run still needs to be verified there.

All timestamps inside memory are relative to the prepared video. Clip labels also show
the corresponding time in the original source. Positive `AUDIO_OFFSET_S` delays audio.
Speech is always transcribed by Whisper; SRT is not used in this workflow.


In [ ]:
from memvid.demo.notebook import DemoSettings, NotebookDemo

PROFILE = "local_small"  # Change to "kaggle" with GPU T4 x2 enabled.
SOURCE_START_S = 0.0
DURATION_S = 180.0  # None for the full video
AUDIO_OFFSET_S = 0.0
BUDGET_TOKENS = 2000
OUTPUT_DIR = REPO_ROOT / "demo_outputs"

settings = DemoSettings(video_path=VIDEO_PATH, audio_path=AUDIO_PATH,
    output_dir=str(OUTPUT_DIR), profile=PROFILE, source_start_s=SOURCE_START_S,
    duration_s=DURATION_S, audio_offset_s=AUDIO_OFFSET_S, budget_tokens=BUDGET_TOKENS)
demo = NotebookDemo(settings)
print("Session:", demo.session)


## 4. Prepare the video and check synchronization

A browser-compatible H264/AAC copy is created for the selected range. Original files
remain unchanged. Preview is limited to ten seconds so notebook output does not contain
the entire long video. Check that speech and visible actions are synchronized.


In [ ]:
from IPython.display import Video, display

demo.prepare()
display(Video(demo.preview(), embed=True, html_attributes='controls preload="metadata"'))


## 5. Build structured memory

Stages: Whisper and frame sampling → visual perception → C3 memory. Locally, SmolVLM
first produces visual captions; Qwen converts those captions and speech into typed
observations. On Kaggle, Qwen3-VL perceives frames and speech directly.

Completed stages are reused when their signatures match. Interrupted captioning/perception
resumes from saved progress. An interrupted memory build restarts C3 from saved observations.
The processing log shows model errors. Failed visual perception blocks building from
transcript-only fallback output. Change settings to create a separate session.


In [ ]:
demo.build()
print("Memory:", demo.run_path)
print("Processing log:", demo.log_path)


## 6. Record IDs and memory operations

| Prefix | Record type | Meaning |
| --- | --- | --- |
| `E`, e.g. `E1` | Entity | A person, object, animal, place or other entity with attributes and aliases. |
| `V`, e.g. `V1` | Event | An action/event, a speech-summary event, or a compressed event summary. |
| `S`, e.g. `S1` | State change | A confirmed attribute transition, recording the old and new values. |
| `R`, e.g. `R1` | Relation | A subject–predicate–object relation referencing entity IDs. |
| `N`, e.g. `N1` | Note | A free-text summary or log entry, mainly used by C1/C2/C4. |

**`V` does not mean visual-only, and `S` does not mean speech.** Speech summaries use
event IDs (`V`) with `modality = "speech"`. The number is a per-type creation counter
within one memory run, not a timestamp, frame number, chunk number or question ID.
An ID can persist through updates; IDs from different runs are not globally unique.

The changes table records `upsert`/`delete` operations and their reasons, with values
before and after each operation. Semantic labels in the underlying journal include
insert, link, update, extend, change, compress and evict. Source chunk numbers are zero-based.


## 7. Inspect memory, ask a question and open supporting clips

Select a chunk to reconstruct memory **as it existed at that step**. Questions retrieve
from that snapshot only. Filter record types, inspect raw visual descriptions and observations,
review before/after updates, and open a record's source. Ask questions in English.

The answer panel shows retrieved records, retrieval scores, used citations and source intervals.
All supporting intervals are available in the clip selector. Long intervals are split into
pieces of at most 30 seconds; only the selected piece is encoded and embedded. Clips are
never padded past the selected memory time. They show provenance, not a guarantee that
the model's interpretation is correct.

Statuses: `answered`, `insufficient_evidence`, `invalid_citations`, `model_error`.
Confidence is a model-reported label, not a calibrated probability.


In [ ]:
demo.display()


## 8. Reopen a saved session

Keep the complete session directory, including `prepared.mp4`, `demo.yaml`, `session.json`,
`derived/` and `runs/`. Opening it relocates its artifact paths and does not rerun preprocessing
or require the original full video. New answers still need the configured model weights
and embedder, from local cache or downloaded on first use. Media tools are needed for new clips.


In [ ]:
# MEDIA_TOOLS = setup_media_tools(REPO_ROOT / ".cache" / "media-tools")
# restored = NotebookDemo.open("/path/to/copied/session_directory")
# restored.display()


## Kaggle verification

Enable Internet and GPU T4 x2. Run setup, select or download media, set `PROFILE = "kaggle"`
and `DURATION_S = None`. Keep outputs under `/kaggle/working`. Check the beginning, middle
and end of memory; ask a cross-segment question and verify clips with audio in the Kaggle browser.
Use the fixed question YAML separately for accuracy evaluation. Do not adapt evaluation questions
to what this memory stores. Raw media, model caches and generated sessions are ignored by Git.
